# Ridge Regression — Optimization

**Goal.** Replace the closed-form ridge solution (2.2 of `02_mathematics.ipynb`) with iterative algorithms — batch gradient descent and a Cholesky-based linear solve — and show that the **regulariser also improves the conditioning** of the optimisation: the condition number drops as $\lambda$ grows, so GD converges *faster* than on the unregularised problem.

**Role of this notebook.** *Algorithms* — pseudocode plus minimal demo code. Math (gradient, Hessian, convexity) is in `02_mathematics.ipynb`; full implementation in `05_hands_on_programming.ipynb`.

**Prerequisites.** `01_linear_regression/03_optimization.ipynb` (Algorithm 1, Theorem 4.2 on convergence rate, the role of $\kappa$); `02_mathematics.ipynb` of this folder (the ridge gradient formula).

**Stage map.** `01_intuition` → `02_mathematics` → **`03_optimization`** → `04_statistics` → `05_hands_on_programming`.

**Five questions.**

1. What is the ridge gradient-descent update rule?
2. Does ridge *help* or *hurt* the conditioning of the optimisation?
3. How does the convergence rate move with $\lambda$?
4. When is the closed form (Cholesky / linear solve) still preferable?
5. How do we sweep an entire path of $\lambda$ efficiently?

In [ ]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

import random
import time

import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import cho_factor, cho_solve

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
rng = np.random.default_rng(SEED)

plt.rcParams["figure.dpi"] = 90

## 1. The ridge gradient-descent update

From eq. (2.1) of `02_mathematics.ipynb`,

> $\nabla$ $L_{\text{ridge}}(\theta)$  =  (2/n) $\cdot$ $X^T (X \theta - y)$  +  2 $\lambda$ $\cdot$ $\theta$.

Plugging into the generic GD step (`01_linear_regression/03_optimization.ipynb` eq. 2.1):

```
ALGORITHM:  Batch gradient descent for ridge regression

Input:   X $\in$ R^{n$\times$p}, y $\in$ R^n, ridge parameter $\lambda$ $\ge$ 0, step size $\eta$ > 0,
         initial $\theta_{0}$, tolerance tol, max iterations K.
Output:  $\theta_{K}$ $\approx$ argmin  $L_{\text{ridge}}(\theta)$.

1.  for k = 0, 1, ..., K - 1:
2.      g_k  $\leftarrow$  (2/n) $\cdot$ $X^T$ (X $\theta_k$ - y)  +  2 $\lambda$ $\cdot$ $\theta_k$
3.      if $\|g_k\|$ $\le$ tol:  return $\theta_k$
4.      $\theta_{k+1}$  $\leftarrow$  $\theta_k$  -  $\eta$ $\cdot$ g_k
5.  return $\theta_{K}$
```

The only change from `01_linear_regression/03_optimization.ipynb` Algorithm 1 is the `+ 2 $\lambda$ $\cdot$ $\theta_k$` on line 2. That single term, however, transforms the geometry of the loss surface, as the next section shows.

## 2. Ridge improves the condition number

From `01_linear_regression/03_optimization.ipynb` §4 the linear-rate constant of GD is ($\kappa$ - 1)/($\kappa$ + 1), where $\kappa$ is the condition number of the Hessian. For ridge:

> $\nabla$^2 L_ridge  =  (2/n) $\cdot$ ($X^T X$ + n $\lambda$ $I_p$).

Substituting the SVD X = U $\Sigma$ $V^T$ (`02_mathematics.ipynb` §3.1), this matrix has eigenvalues

> $\mu_{j}$($\lambda$)  =  (2/n) $\cdot$ ($\sigma_j$^2 + n $\lambda$),    j = 1, …, p.   (2.1)

Hence its condition number is

> $\kappa$($\nabla$^2 L_ridge)  =  ( $\sigma_1$^2 + n $\lambda$ ) / ( $\sigma_{p}$^2 + n $\lambda$ ).   (2.2)

Three observations:

1. **At $\lambda$ = 0** the formula reduces to $\sigma_1$^2 / $\sigma_{p}$^2 = $\kappa$(X)^2 (the OLS condition number from `01_linear_regression/03_optimization.ipynb` §4).
2. **At $\lambda$ → $\infty$** the formula tends to 1 — perfectly conditioned, GD reaches the optimum in essentially one step.
3. **At any $\lambda$ > 0** the ratio is strictly *smaller* than $\kappa$(X)^2 whenever $\sigma_{p}$ > 0. When $\sigma_{p}$ $\approx$ 0 the OLS $\kappa$ is enormous but ridge keeps it finite — exactly the situation where OLS would have *no* unique solution.

**Reading.** Ridge does two helpful things at once: it *defines* a unique answer (Theorem 2.2 of `02_mathematics.ipynb`) **and** makes finding that answer easier numerically. Tikhonov regularisation predates statistical machine learning — it was first introduced by Andrey Tikhonov in the 1940s exactly for this conditioning reason in inverse problems.

### 2.1 Demo: $\kappa$($\nabla$^2 L_ridge) as $\lambda$ sweeps

Synthetic design with deliberately small smallest singular value (so OLS is barely-solvable). Plot $\kappa$ as a function of $\lambda$.

In [ ]:
# Build a design whose smallest singular value is small (ill-conditioned for OLS).
n, p = 200, 10
rng_demo = np.random.default_rng(SEED)
U_, _ = np.linalg.qr(rng_demo.normal(size=(n, p)))
V_, _ = np.linalg.qr(rng_demo.normal(size=(p, p)))
sing_vals = np.geomspace(20.0, 0.05, p)   # ratio σ_max / σ_min = 400 → κ(X)² = 160000
X_ill = U_ @ np.diag(sing_vals) @ V_.T
print(f"σ_1(X)  = {sing_vals[0]:.3f}")
print(f"σ_p(X)  = {sing_vals[-1]:.3f}")
print(f"κ(X)²   = {(sing_vals[0] / sing_vals[-1])**2:.1f}")

lams = np.logspace(-6, 2, 60)
kappas = ((sing_vals[0]**2 + n * lams) /
          (sing_vals[-1]**2 + n * lams))

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.loglog(lams, kappas, "o-", color="steelblue")
ax.axhline(1, color="black", ls=":", label="perfect conditioning (κ = 1)")
ax.set_xlabel("λ (log)")
ax.set_ylabel("κ(∇² L_ridge)  (log)")
ax.set_title("Ridge brings the condition number down monotonically")
ax.legend()
plt.show()

**Reading.** Two regimes:

- **Small $\lambda$** (left side) — $\kappa$ $\approx$ $\kappa$(X)^2 $\approx$ 1.6$\cdot$10^5. OLS is recovered, GD crawls.
- **Large $\lambda$** (right side) — $\kappa$ → 1. GD takes essentially one step.

The slope changes once $\lambda$ is big enough that n $\lambda$ dominates $\sigma_{p}$^2 but not yet $\sigma_{1}$^2. That window — roughly 0.001 to 10 for this design — is where ridge has the largest *qualitative* effect.

## 3. Demo: GD converges faster with ridge

Same ill-conditioned design as §2.1; fit the same y by GD at three $\lambda$ values, with the same $\eta$ for all three. The trajectories speak for themselves.

In [ ]:
true_theta = rng_demo.normal(size=p)
noise = rng_demo.normal(0, 0.5, size=n)
y = X_ill @ true_theta + noise

def gd_ridge(X, y, lam, eta, n_iter):
    n_, p_ = X.shape
    theta = np.zeros(p_)
    losses = []
    for _ in range(n_iter):
        grad = (2.0 / n_) * X.T @ (X @ theta - y) + 2.0 * lam * theta
        theta = theta - eta * grad
        losses.append(float(np.mean((X @ theta - y) ** 2) + lam * theta @ theta))
    return theta, np.array(losses)

# Pick η small enough so the worst-conditioned case (λ = 0) stays stable.
# Theorem 4.2 stable bound:  η ≤ 2 / L_smooth = 2 n / (2 (σ_1² + n λ)) = n / (σ_1² + n λ).
eta = 0.6 * n / (sing_vals[0]**2)   # ≈ 60% of stability ceiling at λ = 0
n_iter = 600

fig, ax = plt.subplots(figsize=(7, 4.5))
for lam, color in [(0.0, "crimson"), (1e-2, "orange"), (1.0, "seagreen")]:
    _, losses = gd_ridge(X_ill, y, lam, eta, n_iter)
    final_minus_floor = losses - losses[-1]    # plot "distance from this run's minimum"
    final_minus_floor[final_minus_floor <= 0] = 1e-16
    ax.plot(final_minus_floor, color=color, label=f"λ = {lam}")
ax.set_yscale("log")
ax.set_xlabel("step k")
ax.set_ylabel("L_ridge(θ_k) − L_ridge(θ\\*)")
ax.set_title("More λ → smaller κ → GD reaches its own optimum faster")
ax.legend()
plt.show()

**Reading.** Each curve plots the gap between the current loss and the *limit* of that particular GD run — so all three optimums sit at 1e-16 on the y-axis and we can compare convergence *rates* directly. Higher $\lambda$ (smaller $\kappa$ by (2.2)) → straighter line on the log plot → faster convergence. This is Theorem 4.2 of `01_linear_regression/03_optimization.ipynb` made visible.

## 4. Cholesky path — the right closed-form algorithm

For small to moderate p the closed-form (2.2) of `02_mathematics.ipynb` is the best option *if* you implement it well.

**Wrong way (textbook formula).** Form A := $X^T X$ + n $\lambda$ $I_p$, then `np.linalg.inv(A) @ X.T @ y`. The inv() squares the condition number once more and is wasteful.

**Right way.** A is symmetric positive-definite (Theorem 2.2 of `02_mathematics.ipynb`). Compute its **Cholesky factor** A = LLᵀ once, then solve the two triangular systems via `cho_solve`. Cost: $\approx$ (1/3) p^3 for the factorisation, then $\approx$ p^2 per right-hand side. No inv() ever appears.

```
ALGORITHM:  Cholesky-based ridge solve

1.  G  $\leftarrow$  $X^T$ X       (cost  Θ(n p^2))
2.  b  $\leftarrow$  $X^T$ y       (cost  Θ(n p))
3.  A  $\leftarrow$  G + n $\lambda$ I_p
4.  L  $\leftarrow$  Cholesky(A)        # A = L L^T,  L lower-triangular
5.  Solve  L z = b   for z   # forward substitution
6.  Solve  L^T $\theta$ = z for $\theta$   # back substitution
7.  return $\theta$
```

Best of all: steps 1, 2 do not depend on $\lambda$. **For a path of $\lambda$ values we re-use G and b**, paying only the Cholesky factorisation + two triangular solves at each new $\lambda$ — cheaper than a fresh GD warm-start by a huge margin.

In [ ]:
def ridge_path_cholesky(X, y, lams):
    """Compute θ̂_ridge for every λ in lams. Reuses G = X^T X and b = X^T y."""
    n_, p_ = X.shape
    G = X.T @ X
    b = X.T @ y
    thetas = np.empty((len(lams), p_))
    for i, lam in enumerate(lams):
        A = G + n_ * lam * np.eye(p_)
        c, low = cho_factor(A)
        thetas[i] = cho_solve((c, low), b)
    return thetas

t0 = time.perf_counter()
thetas_path = ridge_path_cholesky(X_ill, y, lams)
elapsed = time.perf_counter() - t0
print(f"Path of {len(lams)} λ values via Cholesky: {elapsed*1000:.1f} ms")
print(f"Average per-λ:                            {elapsed*1000/len(lams):.2f} ms")

In [ ]:
# Sanity: largest coefficient magnitude shrinks with λ.
max_abs = np.max(np.abs(thetas_path), axis=1)
fig, ax = plt.subplots(figsize=(6.5, 4))
ax.loglog(lams, max_abs, "o-", color="steelblue")
ax.set_xlabel("λ (log)")
ax.set_ylabel("max |θ̂_j| (log)")
ax.set_title("Closed-form ridge path: coefficient magnitudes shrink monotonically with λ")
plt.show()

## 5. When to prefer GD over closed-form

Two situations.

**(a) p is too large for an n $\times$ p $\times$ p factorisation.** Cholesky cost is Θ(p^3); GD cost per step is Θ(n p). When p $\approx$ 10^5 (text features, image patches), Cholesky is impractical; GD or stochastic GD is the only option.

**(b) p > n.** The Cholesky path still works (Theorem 2.2 of `02_mathematics.ipynb` guarantees $X^T X$ + n $\lambda$ $I_p$ is PD), but you may prefer the **dual** form which only needs an $n \times n$ linear solve. The dual is the starting point for *kernel ridge regression*, beyond the scope of this folder.

For everything else — and for almost all classical regression problems with p in the hundreds — the closed-form Cholesky path is the right default.

## Takeaway

- **Update.**   The only line that changes from OLS GD is the gradient: `+ 2 $\lambda$ $\cdot$ $\theta_k$`.
- **Conditioning.**   $\kappa$($\nabla$^2 L_ridge) = ($\sigma_1$^2 + n $\lambda$) / ($\sigma_{p}$^2 + n $\lambda$) $\in$ [1, $\kappa$(X)^2], strictly less than $\kappa$(X)^2 whenever $\sigma_{p}$ > 0. Ridge always *speeds up* GD.
- **Closed form.**   Cholesky-based solve of ($X^T X$ + n $\lambda$ $I_p$) $\theta$ = Xᵀy is the textbook winner for small-to-moderate p. Pre-compute G = $X^T X$ once and reuse along an entire path of $\lambda$ values.
- **When GD wins.**   p large (≳ 10^5), or kernel-ridge / dual formulations where forming G is infeasible.

Next: `04_statistics.ipynb` — having shown how to *compute* $\hat{\theta}_{ridge}$ cheaply, we study its statistical properties: bias, variance, why ridge has lower MSE than OLS for *some* $\lambda$ > 0 (Theorem of Hoerl & Kennard 1970), and how to pick $\lambda$ via cross-validation.